# P16: QLoRA instruction-tuning of a small LLM for Hindi

Runs on a free GPU:
- **Colab:** Runtime → Change runtime type → T4 GPU.
- **Kaggle:** Settings → Accelerator → GPU T4, and turn **Internet on**.

The fixed train/val/test split is already in the repo (`data/splits/`). Don't regenerate it, so every run uses the same test prompts.

In [ ]:
# Show the GPU. Every run in the report is on one NVIDIA Tesla T4 (16 GB).
# Why it matters: check_run.py fails a run on any other GPU, because peak memory and training time
# are only comparable on the same hardware. Kaggle lists two T4s; train.py and generate.py use only
# GPU 0 (CUDA_VISIBLE_DEVICES=0), so the second one is never used.
!nvidia-smi

## Setup

In [ ]:
# Setup: detect the platform, get the repo at the right branch, install the pinned libraries.
import os, sys, json, re

# Why detect the platform: Colab and Kaggle have different writable folders and different rules for
# what survives the end of a session (see the next cell).
ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/sunny4992mishra/QLoRA-Instruction-DL.git"
REPO = ("/kaggle/working" if ON_KAGGLE else "/content") + "/QLoRA-Instruction-DL"
BRANCH = "main"  # set to a pull request branch to test it before merging

# Clone the first time. On a rerun, fetch and pull instead, so merged changes arrive without a
# second copy of the repo.
if not os.path.exists(REPO):
    !git clone -b {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} fetch origin && git -C {REPO} checkout {BRANCH} && git -C {REPO} pull
# All scripts use paths relative to the repo root (data/splits/..., src/...), so work from there.
%cd {REPO}
# Why pinned versions: transformers, peft, accelerate, bitsandbytes and datasets are fixed in
# requirements.txt, so every team member trains with the same code and the runs are comparable.
# torch is not pinned, because Colab and Kaggle ship their own CUDA build of it.
!pip install -q -r requirements.txt

In [ ]:
# Save adapters and outputs where they survive a disconnect.
# Colab: Google Drive. Kaggle: /kaggle/working (kept when you "Save Version").
# Why not the repo folder on Colab: the session disk is wiped when the runtime disconnects, and a
# full-data run takes about 30 minutes, so its adapter and answers must be written somewhere safe.
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE = "/content/drive/MyDrive/p16"
elif ON_KAGGLE:
    SAVE = "/kaggle/working/p16"
else:
    SAVE = "outputs"
# runs/ gets one folder per training run (adapter, checkpoints, run_info.json); results/ gets one
# answers file per system. Copy results/*.jsonl and runs/*/run_info.json into the repo to commit
# them; adapters stay out of git because they are large.
os.makedirs(f"{SAVE}/runs", exist_ok=True)
os.makedirs(f"{SAVE}/results", exist_ok=True)

# The base model of every run: Apache-2.0, not gated, already instruction-tuned, and small enough
# (1.54B parameters) to train in 4-bit on a 16 GB T4 with 1,024-token sequences.
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
# TAG starts every run name, e.g. Qwen2.5-1.5B-Instruct_r16_n775. The same name is used for
# runs/<name>/, results/<name>.jsonl and results/judge/<name>.jsonl, which is how analyze.py
# joins the three.
TAG = MODEL.split("/")[-1]
# Number of training rows (775): one JSON object per line. A full-data run is named n775.
N_TRAIN = sum(1 for _ in open("data/splits/train.jsonl", encoding="utf-8"))
# Cleaning and split counts from prepare_data.py: 1,153 raw rows, 115 dropped (no Devanagari in
# the prompt), 22 duplicates, 1,016 clean; 775 train / 61 validation rows, 150 test prompts.
print(json.load(open("data/split_stats.json")))

## 0. Smoke test (once per new setup, about 10–20 minutes)

Checks the whole pipeline on a small scale before any real run: the pinned library versions, 4-bit loading, 10 training steps on the first 100 rows, and answers to 8 test prompts with and without the adapter.
It writes to `outputs/smoke/` on the session's own disk, not to `SAVE`, and deletes the previous smoke test first.

To test a pull request before it is merged, set `BRANCH` in the setup cell to the PR's branch name and run from the top.

A good run on a T4 looks like this:

| Output | Expected |
|---|---|
| Version check | every line ends in `ok` |
| `train tokens:` line | `'rows': 100, 'kept': 98, 'truncated': 11, 'dropped_prompt_too_long': 2` |
| `trainable params:` line | `18,464,768`, about 1.2% of the model |
| Check cell | every line starts with `PASS` |

The zero-shot answers should be Hindi text. After only 10 steps, the fine-tuned answers will look much like the zero-shot ones.
If anything fails, paste the full error into issue #1. If everything passes, post the check cell's output there.

In [ ]:
# Installed versions vs requirements.txt
# Why check: a different transformers or peft version can change the chat template, tokenization
# or the trainable-parameter count, and check_run.py would then fail the run. "==" must match
# exactly; ">=" (torch) only needs to be at least that version.
from importlib.metadata import version
from packaging.version import Version

for line in open("requirements.txt"):
    # Split a line such as "peft==0.21.2" into name, operator and version. Comment and blank lines
    # don't match, so they are skipped.
    m = re.match(r"([\w-]+)(==|>=)(\S+)", line.strip())
    if m:
        name, op, want = m.groups()
        have = version(name)
        # Version() compares version numbers properly ("2.10" > "2.6"), which plain string
        # comparison would get wrong.
        ok = have == want if op == "==" else Version(have) >= Version(want)
        print(f"{name:13} {have:14} {'ok' if ok else 'MISMATCH, need ' + op + want}")

In [ ]:
# Smoke test: the whole pipeline at a small scale (10-20 minutes) before a real run (~30 minutes).
# Why outputs/smoke and not SAVE: test files must never mix with real results.
# Why rm -rf first: generate.py skips prompts already in its output file, so a file left from an
# earlier test would make this test do nothing.
SMOKE = "outputs/smoke"
!rm -rf {SMOKE}
# 1. Zero-shot answers to the first 8 test prompts: checks 4-bit loading and batched generation.
!python src/generate.py --model {MODEL} --limit 8 --out {SMOKE}/zeroshot.jsonl
# 2. Only 10 optimizer steps (--max-steps 10) on the first 100 rows at r = 16: checks tokenization,
#    loss masking, LoRA, the 8-bit optimizer and run_info.json. Expected "train tokens" line:
#    100 rows, 98 kept, 11 truncated, 2 dropped (prompt alone >= 1,024 tokens).
!python src/train.py --model {MODEL} --rank 16 --train-size 100 --max-steps 10 --output-dir {SMOKE}/r16
# 3. The same 8 prompts with the new adapter: checks that an adapter loads on the 4-bit base model.
!python src/generate.py --model {MODEL} --adapter {SMOKE}/r16/adapter --limit 8 --out {SMOKE}/r16.jsonl

In [ ]:
# Check the smoke test's outputs. Every line should start with PASS.
import math

info = json.load(open(f"{SMOKE}/r16/run_info.json"))
gpu = info["gpu_memory_peak_training"]
answers = {name: [json.loads(l) for l in open(f"{SMOKE}/{name}.jsonl", encoding="utf-8")]
           for name in ("zeroshot", "r16")}
fp16_gpu = any(n in gpu.get("gpu", "") for n in ("T4", "P100"))  # no native bf16 on these
checks = {
    # QLoRA needs the base model in 4-bit NF4; utils.py turns 4-bit off without a CUDA GPU.
    "loaded in 4-bit": info["quantized_4bit"],
    # The T4 has no native bf16, so compute must be fp16 there (utils.compute_dtype).
    "fp16 on T4/P100, bf16 on newer GPUs": info["compute_dtype"] == ("torch.float16" if fp16_gpu else "torch.bfloat16"),
    # --max-steps 10 overrides the 3 epochs.
    "10 training steps": info["global_steps"] == 10,
    # A NaN or infinite loss would mean broken fp16 training or a masking bug (e.g. no response tokens).
    "train and val loss are finite": all(math.isfinite(x) and x > 0 for x in (info["final_train_loss"], info["final_eval_loss"])),
    # Peak allocated memory is the "peak GPU memory" number in the report.
    "GPU memory recorded": "peak_allocated_gb" in gpu,
    "8 answers in each file": all(len(rows) == 8 for rows in answers.values()),
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", name)

print("\nGPU memory after load:", info["gpu_memory_after_load"])
print("GPU memory peak:", gpu)
print("train / val loss:", round(info["final_train_loss"], 3), "/", round(info["final_eval_loss"], 3))
# Time per optimizer step without the validation passes, to estimate how long a full run takes.
evals = [h["eval_runtime"] for h in info["log_history"] if "eval_runtime" in h]
per_step = (info["train_seconds"] - sum(evals)) / info["global_steps"]
# Full r=16 run: 766 kept rows / 16 per step = 48 steps per epoch, x 3 epochs, 3 validations.
print(f"{per_step:.0f} s per training step -> rough estimate for a full run: "
      f"{(144 * per_step + 3 * max(evals, default=0)) / 60:.0f} min")

# Share of letters that are Devanagari (Unicode U+0900 to U+097F). Above 0.5 counts as "mostly
# Hindi", the same rule as answer_stats.py.
deva = lambda t: sum("\u0900" <= c <= "\u097F" for c in t) / max(1, sum(c.isalpha() or "\u0900" <= c <= "\u097F" for c in t))
for name, rows in answers.items():
    print(f"{name}: {sum(deva(r['output']) > 0.5 for r in rows)} of {len(rows)} answers mostly Devanagari")
# One example. After only 10 steps the fine-tuned answer is expected to look much like zero-shot;
# this is a pipeline check, not a quality check.
print("\nPROMPT:", answers["zeroshot"][0]["instruction"][:300])
print("\nZERO-SHOT:", answers["zeroshot"][0]["output"][:400])
print("\nAFTER 10 STEPS:", answers["r16"][0]["output"][:400])

## 1. Zero-shot baseline

Base model without fine-tuning, on the 150 held-out test prompts. Add `--limit 20` for a quick check first.
If the session dies, re-run the same cell: prompts already in the output file are skipped.

In [ ]:
# Zero-shot baseline: the same 4-bit base model with no adapter, so every gain is measured against it.
# It uses the same prompt format (prompts.py), greedy decoding, repetition penalty 1.1 and the
# 512-token limit as the fine-tuned runs, so the only difference is the adapter.
# Result in the report: 103 of 150 answers run to the 512-token limit and 79 loop; judge mean 1.63.
# If the session dies, rerun the cell: prompts already in the output file are skipped.
!python src/generate.py --model {MODEL} --out {SAVE}/results/{TAG}_zeroshot.jsonl

## 2. QLoRA fine-tuning (one run)

`N = 0` uses all training rows. `N = 250` uses the first 250 (subsets are nested: 100 ⊂ 250 ⊂ 500).

In [ ]:
# One QLoRA run. Only the rank and the number of rows change between runs; every other setting is
# the train.py default (lr 2e-4, batch 2 x 8 = 16, 3 epochs, 1,024 tokens, seed 42, alpha = 2 x rank),
# and check_run.py fails a run that changes any of them.
# N = 0 means all 775 rows. N = 100, 250 or 500 takes the first N rows of train.jsonl, which is in a
# fixed random order, so the subsets are nested (100 in 250 in 500).
RANK, N = 16, 0
# Run name, e.g. Qwen2.5-1.5B-Instruct_r16_n775.
NAME = f"{TAG}_r{RANK}_n{N or N_TRAIN}"
RUN = f"{SAVE}/runs/{NAME}"
# Writes adapter/ (the end of epoch 3, the one evaluated), checkpoint-* after each epoch and
# run_info.json. Expected for r = 16 on all rows: 766 rows kept, 18,464,768 trainable parameters
# (1.18%), 144 steps, about 30 minutes and about 5.94 GB peak memory on a T4.
!python src/train.py --model {MODEL} --rank {RANK} --train-size {N} --output-dir {RUN}

In [ ]:
# Read back the run's record. These fields are the source of the cost and validation-loss tables.
info = json.load(open(f"{RUN}/run_info.json"))
print({k: info[k] for k in ["trainable_params", "trainable_pct", "gpu_memory_peak_training",
                             "train_seconds", "final_train_loss", "final_eval_loss"]})
# Validation loss after each of the 3 epochs (61 validation rows). In every full-data run it was
# lowest after epoch 2 and rose in epoch 3 (r16: 0.951, 0.925, 0.937). The epoch-3 adapter is still
# the one evaluated, so all runs are treated the same.
print("val loss per epoch:", [(round(h["epoch"], 2), round(h["eval_loss"], 4))
                              for h in info["log_history"] if "eval_loss" in h])

## 3. Generate test answers with the adapter

In [ ]:
# Answers from the fine-tuned model: the same 4-bit base model plus the adapter from the end of
# epoch 3, with the same decoding settings as the zero-shot baseline.
# The output file is named after the run, so check_run.py, judge.py and analyze.py find it.
!python src/generate.py --model {MODEL} --adapter {RUN}/adapter --out {SAVE}/results/{NAME}.jsonl

## 4. Quick sanity check

Share of answers written mostly in Devanagari, and one example side by side. This is only a sanity check; the real evaluation is the LLM judge + human ratings.

In [ ]:
# Why only a sanity check: the share of Devanagari says whether an answer is in Hindi, not whether
# it is right. Correctness is scored later by the LLM judge (judge.py) and the human raters.
def load(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")]

# Share of letters in the Devanagari block (U+0900 to U+097F). Digits and punctuation don't count.
def devanagari_share(text):
    letters = [c for c in text if c.isalpha() or "\u0900" <= c <= "\u097F"]
    return sum("\u0900" <= c <= "\u097F" for c in letters) / max(1, len(letters))

# Zero-shot is already 148 of 150 mostly Devanagari, so the base model's problem is stopping on
# time, not writing in the wrong language.
for name in [f"{TAG}_zeroshot", NAME]:
    path = f"{SAVE}/results/{name}.jsonl"
    if os.path.exists(path):
        rows = load(path)
        hindi = sum(devanagari_share(r["output"]) > 0.5 for r in rows) / len(rows)
        print(f"{name}: {len(rows)} answers, {hindi:.0%} mostly Devanagari")

# One test prompt side by side: prompt, zero-shot, fine-tuned and the reference (first 600 characters).
# Both files are in test-set order, so index i is the same prompt in both.
base, tuned = load(f"{SAVE}/results/{TAG}_zeroshot.jsonl"), load(f"{SAVE}/results/{NAME}.jsonl")
i = 0
print("PROMPT:\n", base[i]["instruction"], "\n\nZERO-SHOT:\n", base[i]["output"][:600],
      "\n\nFINE-TUNED:\n", tuned[i]["output"][:600], "\n\nREFERENCE:\n", base[i]["reference"][:600])

## 5. Experiment grid

Rank sweep at full data, then data-size sweep at a fixed rank. Each run saves to `SAVE`; re-running the cell skips runs that already have `run_info.json`.
A run cut off mid-training restarts from scratch, so split the grid across team members / sessions.

In [ ]:
# The full grid: the rank sweep r = 4, 8, 16, 32, 64 on all rows (n = 0), then the data-size sweep
# N = 100, 250, 500 at r = 16. r = 16 on all rows belongs to both sweeps, so 8 runs in total.
# Why one variable at a time: any difference between two runs is then due to the rank or the data
# size alone.
GRID = [(r, 0) for r in (4, 8, 16, 32, 64)] + [(16, n) for n in (100, 250, 500)]

for rank, n in GRID:
    name = f"{TAG}_r{rank}_n{n or N_TRAIN}"
    run = f"{SAVE}/runs/{name}"
    # Why skip runs that have run_info.json: train.py writes it only at the very end, so a finished
    # run is never trained twice. A run cut off mid-training has none and starts again from scratch.
    if not os.path.exists(f"{run}/run_info.json"):
        !python src/train.py --model {MODEL} --rank {rank} --train-size {n} --output-dir {run}
    # generate.py is always called, because it resumes: prompts already answered are skipped.
    !python src/generate.py --model {MODEL} --adapter {run}/adapter --out {SAVE}/results/{name}.jsonl

In [ ]:
# Summary table: one row per finished run.
# A quick look only; the report's tables and figures come from src/analyze.py.
rows = []
for name in sorted(os.listdir(f"{SAVE}/runs")):
    path = f"{SAVE}/runs/{name}/run_info.json"
    if os.path.exists(path):
        i = json.load(open(path))
        # rows_after_trunc: rows left after the 1,024-token limit (766 of 775; 98, 248, 496 for
        # N = 100, 250, 500). peak_gpu_gb is peak allocated memory, from 5.83 GB at r = 4 to 6.43 GB
        # at r = 64. val_loss is the epoch-3 validation loss.
        rows.append({"run": name, "rank": i["args"]["rank"], "train_rows": i["train_tokens"]["rows"],
                     "rows_after_trunc": i["train_tokens"]["kept"],
                     "trainable_params": i["trainable_params"],
                     "peak_gpu_gb": i["gpu_memory_peak_training"].get("peak_allocated_gb"),
                     "train_min": round(i["train_seconds"] / 60, 1), "val_loss": i["final_eval_loss"]})
# pandas is used only here, to display the table; the scripts in src/ don't need it.
import pandas as pd
pd.DataFrame(rows)